cuPFGrowth

A short tutorial for cuPFGrowth. See the [algorithm guide](https://github.com/udayRage/cuPAMI/blob/main/docs/algorithms/cuPFGrowth.md) for details.

In Colab, choose Runtime → Change runtime type → GPU. Install the library from GitHub and a CuPy wheel matching your CUDA installation. The setup below uses CUDA 12. Run setup once; skip it if your local environment is already configured.

In [1]:
import importlib.util
import subprocess
import sys

if importlib.util.find_spec("cupami") is None:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "git+https://github.com/udayRage/cuPAMI.git"])
if importlib.util.find_spec("cupy") is None:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "cupy-cuda12x"])

In [2]:
import cupy as cp

print("CUDA devices:", cp.cuda.runtime.getDeviceCount())


CUDA devices: 2


Create a small dataset

Use TemporalDatabase with a fixed seed so the input can be reproduced.

In [3]:
from cupami import TemporalDatabase, cuPFGrowth

generator = TemporalDatabase(transactions=100, items=8, avg_length=3, seed=42)
data = generator.create()
print(data[:5])


[[1, 2, 5, 6, 7], [2, 1, 2, 4, 5], [3, 1], [4, 2, 4, 7], [5, 1, 2, 7]]


Mine patterns

`minSup=0.1` means 10% of transactions. `maxPer=30` allows gaps of at most 30 timestamp units, including the start and end gaps.

In [4]:
miner = cuPFGrowth(data, minSup=0.1, maxPer=30)
miner.mine()


View results

Show the first ten patterns and their supports.

In [5]:
from itertools import islice

patterns = miner.getPatterns()
print("Patterns:", len(patterns))
for pattern, support in islice(patterns.items(), 10):
    print(pattern, support)


Patterns: 33
4 (29, 13)
3	4 (10, 20)
1	4 (11, 25)
4	8 (12, 30)
2	4 (12, 25)
4	6 (13, 21)
3 (34, 9)
3	5 (12, 21)
3	7 (13, 24)
3	8 (14, 28)


Timing and output

Times are in seconds. Save the patterns to a local text file.

In [6]:
print("Loading:", miner.getLoadingTime())
print("Mining:", miner.getMiningTime())
print("Total:", miner.getTotalRuntime())
miner.save("patterns.txt")


Loading: 0.0004001809284090996
Mining: 0.2741712441202253
Total: 0.2745714250486344
